In [ ]:
import numpy as np 

def sample_uniform_points_in_triangles(mesh, m_samples, rng=None):
    """
    Return samples using reflection trick
    """

    if rng is None:
        rng = np.random.defaut_rng()

    # triangle vertex coordinates
    p = mesh.p.T # (n_vertices, 2)
    t = mesh.t.T # (n_elements, 3)
    v0 = p[t[:, 0]]
    v1 = p[t[:, 1]]
    v2 = p[t[:, 2]]

    # reflection trick on reference triangle
    u = rng.random((t.shape[0], m_samples))
    v = rng.random((t.shape[0], m_samples))
    mask = (u + v) > 1.0
    u[mask] = 1.0 - u[mask]
    v[mask] = 1.0 - v[mask]

    # barycentric weights on reference triangle are (0,0), (1,0) and (0,1)
    w0 = 1.0 - u - v
    w1 = u
    w2 = v 

    # affine map to other triangles
    pts = (
        w0[..., None] * v0[:, None, :]
        + w1[..., None] * v1[:, None, :]
        + w2[..., None] * v2[:, None, :]
    )
    return pts  # (n_elements, m_samples, 2)


Next we build the **randomized projection operator** 

In [18]:
def pi_hat_0(mesh, load_func, m_samples = 10, rng=None):
    """ 
    Compute 0 order random projection 
    """
    pts = sample_uniform_points_in_triangles(mesh, m_samples, rng=rng)
    x = pts[..., 0]
    y = pts[..., 1]
    vals = load_func(x, y)
    vals = np.asarray(vals)
    if vals.ndim == 0:
        vals = np.full_like(x, float(vals), dtype=float)
    fhat_K = np.mean(vals, axis = 1)
    return fhat_K

Next assemble the load vector

In [19]:
from skfem import *
from skfem.helpers import grad

def load_func(x, y):
    return 1.0

# define linear form of RHS with test function v
@LinearForm
def load_quad(v, w):
    x, y = w.x
    return load_func(x, y) * v

@LinearForm
def load_mc(v, w):
    return w.fK * v

def assemble_load_mc_P0(basis, fhat_K):
    """ 
    Assemble the load vector for the RHS = piecewise constant on each element K
    """
    
    return asm(load_mc, basis, fK=fhat_K)

# Residual estimator with input mesh m and approximation u
def eval_estimator(m, u):
    # interior residual
    # Basis elements contain the mesh and the discretization space
    basis = Basis(m, e)

    # implement residual
    @Functional
    def interior_residual(w):
        h = w.h
        x, y = w.x
        return h**2 * load_func(x, y)**2
    
    # return residual for each element depending on basis and the interpolation of u
    eta_K = interior_residual.elemental(basis, w = basis.interpolate(u))

    # get the solution values of bith sides of a triangle from each side (may jump)
    fbasis = [InteriorFacetBasis(m, e, side=i) for i in [0,1]]
    w = {'u' + str(i + 1): fbasis[i].interpolate(u) for i in [0, 1]}

    # Compute jump term  η_e^2​≈h_e​∫_e​((∇u_h​∣K1​​−∇u_h​∣K2​​)⋅n)^2 ds
    @Functional
    def edge_jump(w):
        h = w.h
        n = w.n
        dw1 = grad(w['u1'])
        dw2 = grad(w['u2'])
        return h * ((dw1[0] - dw2[0]) * n[0] +
                    (dw1[1] - dw2[1]) * n[1])**2

    # get the values per interior facet
    eta_E = edge_jump.elemental(fbasis[0], **w)

    # as each interior facet belongs to two elements, add half its conribution to each adjacent element
    tmp = np.zeros(m.facets.shape[1])
    np.add.at(tmp, fbasis[0].find, eta_E)
    eta_E = np.sum(0.5 * tmp[m.t2f], axis=0)

    return eta_K + eta_E

"""
Implement the Dörfler marking strategy
"""
def dorfler_marking(eta2, theta = 0.5):
    """
    Returns element indices to refine
    Assumes the squared residual error
    """
    eta2 = np.asarray(eta2)
    total = eta2.sum()

    if total <= 0:
        return np.array([], dtype = int)
    
    order = np.argsort(eta2)[::-1] # sort elements by decreasing error value
    csum = np.cumsum(eta2[order]) # compute the sum cumulative sums
    target = theta * total

    k = np.searchsorted(csum, target) + 1 # find first index where csum is larger than target
    marked = order[:max(1, k)] # take first k element indices for the marking
    return marked

## AFEM loop

In [20]:
from skfem import *
from skfem.models.poisson import laplace
from skfem.visuals.matplotlib import draw, plot
import numpy as np

m = MeshTri.init_sqsymmetric().refined(2)
e = ElementTriP1()


n_refinements = 10
theta = 0.5

rhs_mode = "mc" # quad or mc
m_samples = 5   # number of MC samples per element
rng = np.random.default_rng(0) # seed for reproducibility

for itr in reversed(range(n_refinements)):
    basis = Basis(m, e)

    K = asm(laplace, basis)

    if rhs_mode == "quad":
        f = asm(load_quad, basis)
    elif rhs_mode == "mc":
        fhat_K = pi_hat_0(m, load_func, m_samples, rng)
        f = assemble_load_mc_P0(basis, fhat_K)
    else:
        raise ValueError("rhs_mode must be either 'quad' or 'mc'")
    
    I = m.interior_nodes()
    u = solve(*condense(K, f, I=I))

    if itr > 0:
        eta2 = eval_estimator(m, u)
        marked = dorfler_marking(eta2, theta=theta)
        m = m.refined(marked).smoothed()

ax = draw(m)
plot(m, u, ax = ax, shading = 'gouraud', colorbar = True).show()

ValueError: Input array has wrong size.